<div align="center">
    <img src="https://www.sharif.ir/documents/20124/0/logo-fa-IR.png/4d9b72bc-494b-ed5a-d3bb-e7dfd319aec8?t=1609608338755" alt="Logo" width="200">
    <p><b>Sharif University of Technology</b></p>
    <p>Deep Learning Course, Dr. Soleymani</p>
    <p>Spring 2026</p>
    <p><b>Practical Homework 5 — Self-Supervised Learning (SimCLR)</b></p>
</div>

---

# Self-Supervised Learning with SimCLR — Homework Notebook

In this assignment you will implement **SimCLR** (*A Simple Framework for Contrastive Learning of Visual Representations*) in PyTorch, step by step.

The notebook is split into 9 sections. Each section starts with a short explanation, occasionally asks a theoretical question, and then contains one or more **TODO** code cells for you to complete. Read the comment at the top of every TODO cell carefully — it tells you exactly what is expected.

## Outline

1. **Introduction** — what is contrastive learning and why it works.
2. **Setup** — imports, device configuration, helper functions.
3. **Data Augmentation** — the SimCLR augmentation pipeline.
4. **Dataset & DataLoader** — generating two views per unlabelled image.
5. **Model Architecture** — ResNet backbone + projection head.
6. **Contrastive Loss** — the NT-Xent (InfoNCE) loss.
7. **Training Loop** — putting everything together.
8. **Linear Evaluation** — freezing the encoder and training a linear classifier on top.
9. **Discussion** — analysing the results.

> **How to submit:** Run every cell from top to bottom in order, answer all theoretical questions in markdown cells right after the question, and make sure all TODOs are completed before exporting.

## 1. Introduction

Supervised deep learning requires massive labelled datasets, which are expensive and time-consuming to create.
Self-supervised learning (SSL) is a paradigm where a model learns representations from the data itself, without external labels. One of the most successful SSL approaches is **contrastive learning**.

### The core idea

We take an image $x$ and create two random augmentations (views) $\tilde{x}_i$ and $\tilde{x}_j$.
These two views form a **positive pair** — they come from the same underlying image and should have similar feature representations.
All other images in the batch are treated as **negative examples**.

A neural network encoder $f(\cdot)$ followed by a small projection head $g(\cdot)$ maps each view to an embedding $z = g(f(\tilde{x}))$.
The loss function encourages embeddings of a positive pair to be close (similar) while pushing apart embeddings of negative examples.

### SimCLR in one paragraph

SimCLR (Chen et al., 2020) combines: (i) a strong **stochastic augmentation** module (random crop, colour jitter, random flip, Gaussian blur), (ii) a **base encoder** (ResNet-18/50), (iii) a small **projection head** (2-layer MLP), and (iv) a **contrastive loss** computed on the L2-normalised embeddings — trained with large batches and a cosine-annealing learning rate.

> **Question 1.** Why do we need a projection head $g(\cdot)$ on top of the encoder, instead of computing the loss directly on encoder features $h = f(\tilde{x})$?
>
> **Question 2.** What is the role of the temperature parameter $\tau$ in the NT-Xent loss?

<div dir="rtl" align="right">

### پاسخ سؤال ۱

دلیل استفاده از projection head این است که فضای یادگیری کنتراستی را از ویژگی‌های اصلی encoder جدا کنیم. اگر loss را مستقیماً روی خروجی encoder یعنی (h) حساب کنیم، ممکن است encoder مجبور شود بخشی از اطلاعات مفید تصویر را فقط برای بهتر شدن loss حذف کند.

در حالی که بعضی ویژگی‌ها مثل رنگ، جزئیات ظاهری یا بافت تصویر ممکن است برای کارهای بعدی مفید باشند، حتی اگر در تشخیص دو augmentation از یک تصویر اهمیت زیادی نداشته باشند.

projection head این امکان را می‌دهد که loss روی فضای (z) اعمال شود و encoder بتواند ویژگی‌های عمومی‌تر و مفیدتری یاد بگیرد. بعد از پایان آموزش هم معمولاً projection head را کنار می‌گذاریم و از خروجی خود encoder استفاده می‌کنیم.

### پاسخ سؤال ۲

پارامتر دما یا   مشخص می‌کند که مدل تا چه حد به تفاوت شباهت بین نمونه‌های مثبت و منفی حساس باشد.

وقتی مقدار  tau  کوچک باشد، تفاوت شباهت‌ها پررنگ‌تر می‌شود و مدل روی نمونه‌های منفی سخت‌تر تمرکز بیشتری می‌کند. در نتیجه اگر دو نمونه مثبت به اندازه کافی شبیه نباشند، جریمه بیشتری دریافت می‌کنند.

وقتی مقدار  tau بزرگ‌تر باشد، توزیع شباهت‌ها نرم‌تر می‌شود و تفاوت بین نمونه‌های مثبت و منفی تأثیر کمتری روی loss می‌گذارد.

در مجموع،  tau  میزان سخت‌گیری مدل در جدا کردن نمونه‌های مثبت از نمونه‌های منفی را کنترل می‌کند.

</div>


## 2. Setup

Below is the only cell in this section. The imports are already filled in (so you can focus on the SimCLR-specific parts), but you must complete two small TODOs: setting a reproducibility seed and selecting the device.

In [1]:
# -------------------- Setup --------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast

import numpy as np
import os
import sys
from tqdm import tqdm
import logging
from datetime import datetime

# TODO 2.1 — Reproducibility.
# Set the random seed for both `torch` and `numpy` to 0 so that runs are
# repeatable. (One line each.)
# Hint: torch.manual_seed(...) and np.random.seed(...)
torch.manual_seed(0)
np.random.seed(0)

# TODO 2.2 — Device selection.
# Pick CUDA if a GPU is available, otherwise CPU, and store the result in
# the variable `device` so the rest of the notebook can move tensors there.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Using device: cuda


## 3. Data Augmentation

SimCLR uses a carefully designed augmentation pipeline that creates two *independent* random views of every image:

- **Random Resized Crop** — random crop + resize to a fixed size.
- **Random Horizontal Flip** — probability 0.5.
- **Colour Jitter** — random changes to brightness, contrast, saturation, hue (strength scaled by $s$).
- **Random Grayscale** — probability 0.2.
- **Gaussian Blur** — smooths the image with a random sigma.

### 3.1 Gaussian Blur

`torchvision` doesn't ship the SimCLR-style Gaussian blur, so we build it ourselves with two separable 1-D convolutions. Given kernel radius $r$ and sigma $\sigma$ sampled uniformly from $[0.1, 2.0]$, the 1-D weights are

$$ w_i \;\propto\; \exp\!\left(-\tfrac{i^2}{2\sigma^2}\right), \qquad i \in \{-r,\dots,r\}, $$

normalised to sum to 1. We then apply the horizontal kernel, then the vertical kernel.

> **Question 3.** Why is it important to randomly sample $\sigma$ per image rather than fixing it? What does this stochasticity buy us in contrastive learning?

**Your answer to Q3 (random $\sigma$):**

<div dir="rtl" align="right">


اگر مقدار  sigma  را ثابت در نظر بگیریم، همه تصاویر تقریباً با یک شدت مشخص تار می‌شوند و مدل ممکن است به همان الگوی ثابت عادت کند. اما وقتی  sigma برای هر تصویر به‌صورت تصادفی انتخاب شود، میزان تاری در هر بار augmentation متفاوت خواهد بود.

این تصادفی بودن باعث می‌شود دو نمای ساخته‌شده از یک تصویر دقیقاً شبیه هم نباشند و مدل نتواند فقط با تکیه بر جزئیات سطحی یا میزان مشخصی از تاری آن‌ها را تشخیص دهد. در نتیجه، مدل مجبور می‌شود ویژگی‌های مهم‌تر و اصلی‌تر تصویر، مثل شکل و ساختار کلی اشیا را یاد بگیرد.

بنابراین، انتخاب تصادفی  sigma  تنوع داده‌های آموزشی را بیشتر می‌کند، از وابستگی مدل به یک نوع تاری ثابت جلوگیری می‌کند و باعث می‌شود ویژگی‌های یادگرفته‌شده مقاوم‌تر و عمومی‌تر باشند.

</div>





In [2]:
# -------------------- GaussianBlur --------------------
class GaussianBlur:
    """Apply a Gaussian blur with a random sigma to a PIL image (CPU)."""

    def __init__(self, kernel_size: int):
        # Force kernel_size to be odd so it has a well-defined centre.
        radius = kernel_size // 2
        kernel_size = radius * 2 + 1
        self.kernel_size = kernel_size
        self.radius = radius

        # TODO 3.1.a — Build two depthwise (groups=3) convolutions:
        #   self.blur_h : kernel shape (kernel_size, 1)    -> horizontal blur
        #   self.blur_v : kernel shape (1, kernel_size)    -> vertical blur
        # Both should have padding=0 and bias=False.
        self.blur_h = nn.Conv2d(
            in_channels=3,
            out_channels=3,
            kernel_size=(kernel_size, 1),
            padding=0,
            groups=3,
            bias=False
        )

        self.blur_v = nn.Conv2d(
            in_channels=3,
            out_channels=3,
            kernel_size=(1, kernel_size),
            padding=0,
            groups=3,
            bias=False
        )

        # TODO 3.1.b — Wrap reflection padding + the two conv layers in
        # a single nn.Sequential called `self.blur`.
        # Use nn.ReflectionPad2d(radius) so the output keeps the input HxW.
        self.blur = nn.Sequential(
            nn.ReflectionPad2d(radius),
            self.blur_h,
            self.blur_v
        )

        self.pil_to_tensor = transforms.ToTensor()
        self.tensor_to_pil = transforms.ToPILImage()

    def __call__(self, img):
        # Convert PIL -> Tensor and add a batch dimension.
        img_tensor = self.pil_to_tensor(img).unsqueeze(0)

        # TODO 3.1.c — Sample a random sigma uniformly from [0.1, 2.0]
        # and build the normalised 1-D Gaussian weights of length kernel_size.
        # Steps:
        #   1) sigma = np.random.uniform(0.1, 2.0)
        #   2) x = np.arange(-self.radius, self.radius + 1)
        #   3) x = exp(-x^2 / (2 sigma^2)); then normalise so x.sum() == 1
        #   4) torch.from_numpy(x).view(1, -1).repeat(3, 1)  # one row per channel
        sigma = np.random.uniform(0.1, 2.0)
        x = np.arange(-self.radius, self.radius + 1)
        x = np.exp(-(x ** 2) / (2 * sigma ** 2))
        x = x / x.sum()
        x = torch.from_numpy(x.astype(np.float32)).view(1, -1).repeat(3, 1)

        # Copy the 1-D kernel into the two convolution weights.
        # Expected shapes: (3, 1, kernel_size, 1) and (3, 1, 1, kernel_size).
        self.blur_h.weight.data.copy_(x.view(3, 1, self.kernel_size, 1))
        self.blur_v.weight.data.copy_(x.view(3, 1, 1, self.kernel_size))

        with torch.no_grad():
            img_tensor = self.blur(img_tensor).squeeze(0)

        return self.tensor_to_pil(img_tensor)

### 3.2 SimCLR augmentation pipeline

Now we compose the full transform. The strength factor $s$ controls the colour jitter intensity (typically $s=1$ for ImageNet-scale data, $s=0.5$ for smaller datasets).

The recommended pipeline is, in order:

1. `RandomResizedCrop(size)`
2. `RandomHorizontalFlip()`
3. `ColorJitter(0.8s, 0.8s, 0.8s, 0.2s)` wrapped in `RandomApply(p=0.8)`
4. `RandomGrayscale(p=0.2)`
5. Your custom `GaussianBlur(kernel_size = int(0.1 * size))`
6. `ToTensor()`

In [3]:
# -------------------- SimCLR transform --------------------
def get_simclr_pipeline_transform(size: int, s: float = 1.0):
    """Return a torchvision `Compose` implementing the SimCLR augmentation."""

    # TODO 3.2 — Build the colour jitter and assemble the pipeline.
    # 1) color_jitter = transforms.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
    # 2) Compose the 6 steps listed in the markdown above, in order.
    #    Apply the colour jitter with RandomApply(p=0.8).
    color_jitter = transforms.ColorJitter(
        brightness=0.8 * s,
        contrast=0.8 * s,
        saturation=0.8 * s,
        hue=0.2 * s
    )

    data_transforms = transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        GaussianBlur(kernel_size=int(0.1 * size)),
        transforms.ToTensor()
    ])

    return data_transforms

### 3.3 Contrastive-learning view generator

For every image we need **two** augmented views (the positive pair). We wrap the transform in a tiny class whose `__call__` applies the base transform `n_views` times to the *same* PIL input and returns a list of tensors.

In [4]:
# -------------------- ContrastiveLearningViewGenerator --------------------
class ContrastiveLearningViewGenerator:
    """Produce `n_views` independently augmented views of one PIL image."""

    def __init__(self, base_transform, n_views: int = 2):
        self.base_transform = base_transform
        self.n_views = n_views

    def __call__(self, x):
        # TODO 3.3 — Return a list of length self.n_views where each entry is
        # `self.base_transform(x)`. Because the transform is stochastic, the
        # two calls produce two DIFFERENT augmented views of the same image.
        # One-liner is fine (list comprehension).
        return [self.base_transform(x) for _ in range(self.n_views)]

> **Question 4.** Why must the two views be produced by **two independent** random draws of the augmentation? What would go wrong if the *same* deterministic transform were applied to both views?

**Your answer to Q4:**

<div dir="rtl" align="right">


دو نمای یک تصویر باید با دو بار اجرای مستقل augmentation ساخته شوند تا هر کدام تغییرات متفاوتی داشته باشند. مثلاً ممکن است یکی از نماها برش متفاوت، رنگ روشن‌تر یا تاری بیشتری نسبت به نمای دیگر داشته باشد.

در این حالت، مدل یاد می‌گیرد که با وجود تفاوت‌های ظاهری، هر دو تصویر مربوط به یک نمونه اصلی هستند. در نتیجه به‌جای حفظ جزئیات سطحی، ویژگی‌های مهم‌تر و معنایی‌تر تصویر را یاد می‌گیرد.

اگر دقیقاً یک تبدیل یکسان و قطعی روی هر دو نما اعمال شود، دو خروجی تقریباً کاملاً مشابه می‌شوند. در این شرایط مسئله برای مدل بیش از حد ساده می‌شود و مدل می‌تواند بدون یادگیری ویژگی‌های مفید، فقط تصاویر یکسان را به هم نزدیک کند.

بنابراین، مستقل بودن augmentationها باعث می‌شود مدل نسبت به تغییراتی مثل برش، رنگ، جهت و تاری مقاوم شود و نمایش‌های عمومی‌تر و کاربردی‌تری یاد بگیرد.

</div>


## 4. Dataset and DataLoader

SimCLR does not need labels, so we use either **STL-10** (in `split='unlabeled'` mode, 96×96 images) or **CIFAR-10** (32×32 images, labels ignored).

The dataset wraps the `ContrastiveLearningViewGenerator` as its `transform`, so each sample returns `(views, _)` where `views` is a list of `n_views` tensors of shape `[C, H, W]`. The label is ignored.

We use a large batch (e.g. 256 or 512) because SimCLR benefits from many negatives, and `drop_last=True` to keep all batches the same size.

> **Question 5.** Why do we set `drop_last=True` in the `DataLoader`? What could go wrong with a final, smaller batch?

<div dir="rtl" align="right">

**پاسخ سؤال ۵:**

در آموزش SimCLR تعداد نمونه‌های داخل هر batch روی نحوه محاسبه loss تأثیر مستقیم دارد، چون نمونه‌های دیگر batch به‌عنوان مثال‌های منفی استفاده می‌شوند.

اگر آخرین batch کوچک‌تر از بقیه باشد، تعداد نمونه‌های منفی آن کمتر می‌شود و شکل یا اندازه ماتریس شباهت با batchهای قبلی فرق می‌کند. این موضوع ممکن است در بخش‌هایی از پیاده‌سازی که اندازه batch را ثابت فرض کرده‌اند باعث خطا شود یا باعث شود loss آخرین batch رفتار متفاوت و ناپایدارتری داشته باشد.

با قرار دادن `drop_last=True`، آخرین batch ناقص حذف می‌شود و همه batchها اندازه یکسانی دارند. در نتیجه محاسبه loss ساده‌تر، ثابت‌تر و قابل‌اعتمادتر خواهد بود.

</div>


In [5]:
# -------------------- Dataset setup --------------------
class ContrastiveLearningDataset:
    def __init__(self, root_folder: str):
        self.root_folder = root_folder

    @staticmethod
    def get_simclr_pipeline_transform(size: int, s: float = 1.0):
        return get_simclr_pipeline_transform(size, s)

    def get_dataset(self, name: str, n_views: int):
        # TODO 4.1 — Fill in the two dataset factories below.
        # For 'cifar10': datasets.CIFAR10(root, train=True, download=True,
        #                                  transform = ContrastiveLearningViewGenerator(
        #                                      self.get_simclr_pipeline_transform(32), n_views))
        # For 'stl10' : datasets.STL10(root, split='unlabeled', download=True,
        #                                  transform = ContrastiveLearningViewGenerator(
        #                                      self.get_simclr_pipeline_transform(96), n_views))
        valid_datasets = {
            'cifar10': lambda: datasets.CIFAR10(
                root=self.root_folder,
                train=True,
                download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(32),
                    n_views
                )
            ),
            'stl10': lambda: datasets.STL10(
                root=self.root_folder,
                split='unlabeled',
                download=True,
                transform=ContrastiveLearningViewGenerator(
                    self.get_simclr_pipeline_transform(96),
                    n_views
                )
            ),
        }

        if name not in valid_datasets:
            raise ValueError(
                f"Invalid dataset name: {name}. Choose 'stl10' or 'cifar10'."
            )

        return valid_datasets[name]()


# TODO 4.2 — Build the unlabelled training loader.
# Pick `dataset_name`, `batch_size`, `n_views`, then build the loader with:
#   shuffle=True, num_workers=2, pin_memory=True, drop_last=True
dataset_name = 'stl10'          # or 'cifar10'
batch_size = 256
n_views = 2

dataset = ContrastiveLearningDataset('./data')
train_dataset = dataset.get_dataset(dataset_name, n_views)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

print(f"Number of training batches: {len(train_loader)}")
print(
    f"Each batch returns a list of {n_views} views, each of shape:",
    next(iter(train_loader))[0][0].shape,
)

100%|██████████| 2.64G/2.64G [07:14<00:00, 6.07MB/s]


Number of training batches: 390
Each batch returns a list of 2 views, each of shape: torch.Size([256, 3, 96, 96])


## 5. Model Architecture

SimCLR uses a standard ResNet backbone as the encoder $f(\cdot)$ and a small MLP as the projection head $g(\cdot)$. The projection head maps the encoder output (e.g. 512-d for ResNet-18) into a lower-dimensional embedding space (e.g. 128-d). The whole model is simply $g \circ f$.

You will implement `ResNetSimCLR`:

- Take a base model name (`'resnet18'` or `'resnet50'`) and instantiate it from `torchvision.models` **without** ImageNet weights.
- Read the encoder's feature dimension from `backbone.fc.in_features` (call it `dim_mlp`).
- **Replace** `backbone.fc` with a 2-layer projection head:
  $$\texttt{Linear}(\text{dim\_mlp} \to \text{dim\_mlp}) \to \texttt{ReLU} \to \texttt{Linear}(\text{dim\_mlp} \to \text{out\_dim})$$

After contrastive pre-training the projection head is discarded; only the encoder is kept for downstream tasks.

> **Question 6.** Why does the projection head use the same dimension for its hidden layer as the encoder output? (Hint: capacity vs. bottleneck.)

**Your answer to Q6 (projection-head hidden dimension):**

<div dir="rtl" align="right">

[link text](https://)
لایه پنهان projection head همان ابعاد خروجی encoder را نگه می‌دارد تا قبل از رسیدن به embedding نهایی، ظرفیت کافی برای پردازش و ترکیب ویژگی‌ها وجود داشته باشد.

اگر لایه پنهان از همان ابتدا خیلی کوچک باشد، یک bottleneck زودهنگام ایجاد می‌شود و ممکن است بخشی از اطلاعات مفید encoder از بین برود. در این حالت projection head قدرت کافی برای یادگیری یک تبدیل مناسب برای contrastive learning را نخواهد داشت.

با نگه داشتن اندازه لایه پنهان برابر با خروجی encoder، ابتدا یک تبدیل غیرخطی با ظرفیت مناسب انجام می‌شود و سپس در لایه آخر ابعاد به `out_dim` کاهش پیدا می‌کند. بنابراین bottleneck فقط در خروجی نهایی ایجاد می‌شود، نه در میانه مسیر.

</div>


In [6]:
# -------------------- ResNetSimCLR --------------------
class ResNetSimCLR(nn.Module):
    def __init__(self, base_model: str = 'resnet18', out_dim: int = 128):
        super().__init__()

        # TODO 5.1 — Pick the backbone from torchvision.models.
        # Allowed names: 'resnet18' or 'resnet50'.
        # Use num_classes=1000 (the original fc is replaced anyway) and
        # pretrained=False (we want randomly initialised weights).
        self.backbone = self._get_basemodel(base_model)

        # Read the hidden dimension that feeds into the original fc layer.
        dim_mlp = self.backbone.fc.in_features

        # TODO 5.2 — Replace `self.backbone.fc` with the SimCLR projection head:
        #     Linear(dim_mlp -> dim_mlp) -> ReLU -> Linear(dim_mlp -> out_dim)
        self.backbone.fc = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )

    def _get_basemodel(self, model_name: str):
        resnet_models = {
            'resnet18': models.resnet18(pretrained=False, num_classes=1000),
            'resnet50': models.resnet50(pretrained=False, num_classes=1000),
        }

        if model_name not in resnet_models:
            raise ValueError(
                f"Invalid backbone: {model_name}. Choose 'resnet18' or 'resnet50'."
            )

        return resnet_models[model_name]

    def forward(self, x):
        # The forward pass just runs the modified ResNet.
        return self.backbone(x)


# Sanity check — shape should be [2, 128] for STL-10-sized inputs.
model = ResNetSimCLR(base_model='resnet18', out_dim=128).to(device)
dummy = torch.randn(2, 3, 96, 96).to(device)
out = model(dummy)
print(f"Output shape: {out.shape}")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Output shape: torch.Size([2, 128])


> **Question 7.** When we evaluate downstream by freezing the encoder and training only a linear classifier on top, what happens to gradients through the (now removed) projection head? Why is throwing the head away a *reasonable* choice rather than a wasteful one?

**Your answer to Q7:**

<div dir="rtl" align="right">

**پاسخ سؤال ۷:**

در مرحله ارزیابی downstream، projection head حذف می‌شود و encoder نیز ثابت یا freeze می‌شود. بنابراین هیچ گرادیانی از classifier به projection head منتقل نمی‌شود، چون این head دیگر در مدل وجود ندارد. همچنین به دلیل ثابت بودن encoder، وزن‌های آن نیز تغییر نمی‌کنند و فقط پارامترهای linear classifier آموزش داده می‌شوند.

حذف projection head کار منطقی‌ای است، چون این بخش مخصوص حل مسئله contrastive learning ساخته شده است. وظیفه آن این بود که خروجی encoder را به فضایی ببرد که در آن مقایسه نمونه‌های مثبت و منفی و محاسبه contrastive loss بهتر انجام شود.

اما ویژگی‌های خروجی خود encoder معمولاً اطلاعات عمومی‌تر و مفیدتری درباره تصویر نگه می‌دارند. projection head ممکن است بخشی از این اطلاعات را برای سازگار شدن با contrastive loss حذف یا فشرده کند.

بنابراین projection head در زمان پیش‌آموزش نقش مهمی در بهتر یاد گرفتن encoder دارد، حتی اگر بعداً خودش استفاده نشود. به همین دلیل کنار گذاشتن آن هدر دادن نیست؛ می‌توان آن را یک ابزار کمکی برای آموزش بهتر encoder در نظر گرفت.

</div>


## 6. Contrastive Loss — NT-Xent

SimCLR uses the **Normalised Temperature-scaled Cross Entropy** (NT-Xent / InfoNCE) loss.

For a batch of $N$ images we obtain $2N$ embeddings (two views per image). Let $z_i, z_j$ be a positive pair and $\tau$ the temperature. The cosine similarity scaled by $1/\tau$ is

$$ \mathrm{sim}(z_a, z_b) \;=\; \frac{1}{\tau}\, \frac{z_a^\top z_b}{\lVert z_a\rVert \,\lVert z_b\rVert}. $$

For each positive pair $(i,j)$:

$$ \ell(i,j) \;=\; -\log \frac{\exp(\mathrm{sim}(z_i, z_j))}{\sum_{k=1}^{2N} \mathbb{1}_{[k \neq i]} \exp(\mathrm{sim}(z_i, z_k))}. $$

The total loss is averaged over all $2N$ such terms.

In code we build the full $2N \times 2N$ similarity matrix, drop the diagonal (self-similarities), and reshape so each row has 1 positive logit + $(2N-2)$ negative logits. With the positive always at index 0 the loss reduces to plain `CrossEntropyLoss` against the target label `0`.

> **Question 8.** Show that formulating this as cross-entropy over $(2N-1)$ classes — 1 positive + $(2N-2)$ negatives — with the correct class always being the positive, is equivalent to the equation above. *Hint:* expand $-\log \mathrm{softmax}_0(\cdot)$.

**Your answer to Q8 (cross-entropy reformulation):**
<div dir="rtl" align="right">

**پاسخ سؤال ۸:**

برای هر embedding، بعد از حذف شباهت آن با خودش، تعداد (2N-1) مقدار شباهت باقی می‌ماند. یکی از آن‌ها مربوط به جفت مثبت و (2N-2) مورد دیگر مربوط به نمونه‌های منفی هستند.

اگر logit مربوط به جفت مثبت را در ستون صفر قرار دهیم، احتمال کلاس صفر با استفاده از softmax به‌صورت زیر محاسبه می‌شود:

</div>

$
\operatorname{softmax}_0$
========================
$
\frac{\exp(\operatorname{sim}(z_i,z_j))}
{\exp(\operatorname{sim}(z_i,z_j))
+
\sum_{k\neq i,j}\exp(\operatorname{sim}(z_i,z_k))}
$

<div dir="rtl" align="right">

تابع Cross Entropy وقتی برچسب صحیح صفر باشد، منفی لگاریتم همین احتمال را محاسبه می‌کند:

</div>

$
-\log(\operatorname{softmax}_0)$
===============================
$
-\log
\left(
\frac{\exp(\operatorname{sim}(z_i,z_j))}
{\exp(\operatorname{sim}(z_i,z_j))
+
\sum_{k\neq i,j}\exp(\operatorname{sim}(z_i,z_k))}
\right)
$

<div dir="rtl" align="right">

چون مخرج شامل شباهت (z_i) با تمام embeddingهای دیگر به‌جز خودش است، می‌توان عبارت را به شکل زیر نوشت:

</div>

$
-\log
\left(
\frac{\exp(\operatorname{sim}(z_i,z_j))}
{\sum_{k=1}^{2N}
\mathbb{1}_{[k\neq i]}
\exp(\operatorname{sim}(z_i,z_k))}
\right)
$

<div dir="rtl" align="right">

این عبارت دقیقاً با فرمول NT-Xent برای $(\ell(i,j) $)  برابر است.

پس اگر برای هر embedding یک مسئله طبقه‌بندی با (2N-1) کلاس در نظر بگیریم، logit جفت مثبت را در ستون صفر بگذاریم و برچسب همه سطرها را صفر انتخاب کنیم، استفاده از `CrossEntropyLoss` معادل محاسبه مستقیم NT-Xent خواهد بود.

</div>


In [7]:
# -------------------- info_nce_loss --------------------
def info_nce_loss(features, temperature: float = 0.07):
    """
    Args:
        features: Tensor of shape [2 * batch_size, out_dim] — the concatenated
                  embeddings of both views.
        temperature: scalar tau.

    Returns:
        logits: Tensor of shape [2N, 2N-1], where column 0 is the positive.
        labels: Tensor of shape [2N], all zeros (positive at index 0).
    """
    batch_size = features.shape[0] // 2   # N
    device = features.device

    # TODO 6.1 — Build the positive-pair MASK.
    # Construct a vector  [0,1,...,N-1, 0,1,...,N-1]  on `device`.
    # Two positions belong to the same image iff their entries are equal,
    # so `labels = (idx.unsqueeze(0) == idx.unsqueeze(1)).float()` is a
    # [2N, 2N] binary matrix marking positives (including the diagonal).
    idx = torch.arange(batch_size, device=device).repeat(2)
    labels = (idx.unsqueeze(0) == idx.unsqueeze(1)).float()

    # TODO 6.2 — L2-normalise the features so dot products = cosine similarities.
    features = F.normalize(features, dim=1)

    # TODO 6.3 — Compute the full [2N, 2N] similarity matrix (features @ features.T).
    similarity_matrix = features @ features.T

    # Drop the diagonal (self-similarity) from both matrices.
    mask = torch.eye(labels.shape[0], dtype=torch.bool, device=device)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(
        similarity_matrix.shape[0], -1
    )

    # TODO 6.4 — Split positives and negatives.
    # positives : similarity at the locations where the labels mask is True   -> shape [2N, 1]
    # negatives : similarity at the locations where the labels mask is False  -> shape [2N, 2N-2]
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], 1)
    negatives = similarity_matrix[~labels.bool()].view(labels.shape[0], -1)

    # TODO 6.5 — Concatenate so the positive sits at column 0, then divide by tau.
    logits = torch.cat([positives, negatives], dim=1)
    logits = logits / temperature

    # Target label is 0 for every row (positive is at index 0).
    labels = torch.zeros(logits.shape[0], dtype=torch.long, device=device)

    return logits, labels

> **Question 9.** What is the effect of *lowering* the temperature $\tau$? Discuss how it changes (i) the concentration of the softmax distribution and (ii) the magnitude of the gradients with respect to hard negatives.

**Your answer to Q9:**

<div dir="rtl" align="right">

**پاسخ سؤال ۹:**

با کاهش پارامتر دما یا $(\tau)$، مقادیر similarity قبل از ورود به softmax بزرگ‌تر می‌شوند؛ چون همه آن‌ها بر عدد کوچک‌تری تقسیم می‌شوند.

در نتیجه، توزیع softmax متمرکزتر و تیزتر می‌شود. یعنی مدل احتمال بیشتری به نمونه‌ای می‌دهد که بیشترین شباهت را دارد و احتمال نمونه‌های دیگر سریع‌تر کاهش پیدا می‌کند. بنابراین تفاوت کوچک میان شباهت‌ها، تأثیر بیشتری روی خروجی softmax خواهد داشت.

کاهش دما همچنین باعث می‌شود نمونه‌های منفی سخت اهمیت بیشتری پیدا کنند. نمونه منفی سخت، تصویری است که با وجود منفی بودن، شباهت زیادی به نمونه اصلی دارد. چون این نمونه‌ها احتمال softmax بیشتری دریافت می‌کنند، گرادیان بزرگ‌تری نیز ایجاد می‌کنند و مدل با قدرت بیشتری تلاش می‌کند آن‌ها را از نمونه اصلی دور کند.

پس دمای پایین‌تر باعث تمرکز بیشتر روی نمونه‌های دشوار و جداسازی قوی‌تر مثبت‌ها و منفی‌ها می‌شود. البته اگر $(\tau$) بیش از حد کوچک باشد، گرادیان‌ها ممکن است خیلی بزرگ و آموزش ناپایدار شود یا مدل بیش از حد روی تعداد کمی نمونه منفی سخت تمرکز کند.

</div>


## 7. Training Loop

We now glue everything together. Per batch:

1. Concatenate the two views along dim 0 → tensor of shape `[2N, C, H, W]`.
2. Forward through the model → embeddings of shape `[2N, out_dim]`.
3. Compute `logits, labels = info_nce_loss(features, tau)`.
4. Apply `nn.CrossEntropyLoss` to obtain the scalar loss.
5. Backprop with `GradScaler` (mixed precision is optional).
6. Log loss and top-1/top-5 contrastive accuracy every `log_every_n_steps`.
7. Step the cosine-annealing scheduler after each epoch, after a 10-epoch warm-up.

You will fill in three code cells:

- **Cell 28** — the `SimCLR` trainer class.
- **Cell 29** — an `accuracy(...)` helper for top-k accuracy.
- **Cell 30** — the hyperparameter dict + actual training run.

In [8]:
# -------------------- SimCLR Trainer --------------------
class SimCLR:
    def __init__(self, model, optimizer, scheduler, args):
        self.model = model.to(args['device'])
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.args = args
        self.criterion = nn.CrossEntropyLoss().to(args['device'])
        self.scaler = GradScaler(enabled=args['fp16_precision'])

        os.makedirs('./logs', exist_ok=True)
        logging.basicConfig(
            filename=os.path.join(
                './logs',
                f'training_{datetime.now():%Y%m%d_%H%M%S}.log'
            ),
            level=logging.DEBUG,
        )
        logging.info("SimCLR training started.")

    def info_nce_loss(self, features):
        return info_nce_loss(features, self.args['temperature'])

    def train(self, train_loader):
        n_iter = 0
        logging.info(f"Training for {self.args['epochs']} epochs.")

        for epoch_counter in range(self.args['epochs']):
            pbar = tqdm(train_loader, desc=f"Epoch {epoch_counter}")

            for images, _ in pbar:
                # TODO 7.1 — Combine the two views.
                # `images` is a list of two tensors, each [N, C, H, W].
                # Concatenate along dim 0 to get [2N, C, H, W], then move
                # the result to `self.args['device']`.
                images = torch.cat(images, dim=0).to(
                    self.args['device'],
                    non_blocking=True
                )

                with autocast(enabled=self.args['fp16_precision']):
                    # TODO 7.2 — Forward pass and loss.
                    # 1) features = self.model(images)           -> [2N, out_dim]
                    # 2) logits, labels = self.info_nce_loss(features)
                    # 3) loss = self.criterion(logits, labels)
                    features = self.model(images)
                    logits, labels = self.info_nce_loss(features)
                    loss = self.criterion(logits, labels)

                # TODO 7.3 — Backprop with the GradScaler.
                # Sequence: zero_grad, scaler.scale(loss).backward(),
                # scaler.step(optimizer), scaler.update().
                self.optimizer.zero_grad(set_to_none=True)
                self.scaler.scale(loss).backward()
                self.scaler.step(self.optimizer)
                self.scaler.update()

                if n_iter % self.args['log_every_n_steps'] == 0:
                    top1, top5 = accuracy(logits, labels, topk=(1, 5))

                    pbar.set_postfix({
                        'Loss': f'{loss.item():.4f}',
                        'Top1': f'{top1[0].item():.2f}%',
                        'Top5': f'{top5[0].item():.2f}%',
                    })

                    logging.debug(
                        f"Iter {n_iter} | Loss {loss.item():.4f} | "
                        f"Top1 {top1[0].item():.2f}%"
                    )

                n_iter += 1

            # TODO 7.4 — After 10 warmup epochs, step the cosine scheduler
            # once per epoch.
            if epoch_counter >= 10:
                self.scheduler.step()

        logging.info("Training finished.")

        torch.save({
            'epoch': self.args['epochs'],
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }, 'simclr_checkpoint.pth')

        print("Checkpoint saved to simclr_checkpoint.pth")

In [9]:
# -------------------- Top-k accuracy helper --------------------
def accuracy(output, target, topk=(1,)):
    """Compute the top-k accuracy (as percentages) for each k in `topk`."""
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)

        # TODO 7.5
        _, pred = output.topk(
            maxk,
            dim=1,
            largest=True,
            sorted=True
        )
        pred = pred.t()

        correct = pred.eq(target.view(1, -1).expand_as(pred))

        # TODO 7.6
        res = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(
                dim=0,
                keepdim=True
            )
            res.append(correct_k.mul(100.0 / batch_size))

        return res

In [10]:
# -------------------- Training configuration and execution --------------------
# Hyperparameters. Reduce `epochs` for demonstration; SimCLR really wants 200+
# epochs to reach its quoted performance.
args = {
    'device': device,
    'epochs': 10,
    'batch_size': batch_size,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'temperature': 0.07,
    'fp16_precision': False,     # set True if your GPU supports it
    'log_every_n_steps': 50,
    'out_dim': 128,
    'n_views': 2,
}

# TODO 7.7 — Build the model, optimizer and scheduler.
# Model    : ResNetSimCLR(base_model='resnet18', out_dim=args['out_dim']).to(device)
# Optimizer: Adam, learning rate args['lr'], weight_decay args['weight_decay']
# Scheduler: CosineAnnealingLR with T_max = len(train_loader) * args['epochs'],
#            eta_min = 0, last_epoch = -1
model = ResNetSimCLR(
    base_model='resnet18',
    out_dim=args['out_dim']
).to(device)

optimizer = optim.Adam(
    model.parameters(),
    lr=args['lr'],
    weight_decay=args['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=len(train_loader) * args['epochs'],
    eta_min=0,
    last_epoch=-1
)

# TODO 7.8 — Instantiate the trainer and start training.
simclr = SimCLR(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    args=args
)

simclr.train(train_loader)   # uncomment to actually train

/tmp/ipykernel_742/2058944322.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=args['fp16_precision'])
Epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]/tmp/ipykernel_742/2058944322.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.args['fp16_precision']):
Epoch 9: 100%|██████████| 390/390 [10:29<00:00,  1.61s/it, Loss=1.8767, Top1=65.23%, Top5=75.59%]


Checkpoint saved to simclr_checkpoint.pth


In [11]:
print(device)

cuda


> **Question 10.** Why is a *cosine-annealing* learning rate schedule used (after a short warm-up) instead of a constant LR? What would change qualitatively in the loss curves if we kept LR constant?

<div dir="rtl" align="right">

پاسخ سؤال ۱۰:

در ابتدای آموزش، مدل هنوز ویژگی‌های مناسبی یاد نگرفته است و به تغییرات نسبتاً بزرگ در وزن‌ها نیاز دارد. به همین دلیل استفاده از learning rate بزرگ‌تر کمک می‌کند مدل سریع‌تر از وضعیت اولیه فاصله بگیرد. البته warm-up باعث می‌شود learning rate در شروع ناگهان خیلی بزرگ نباشد و آموزش ناپایدار نشود.

بعد از چند epoch، مدل به ناحیه مناسب‌تری از فضای پارامترها می‌رسد. در این مرحله بهتر است learning rate به‌تدریج کاهش پیدا کند تا تغییر وزن‌ها کوچک‌تر و دقیق‌تر شوند. زمان‌بندی cosine این کاهش را به‌صورت نرم انجام می‌دهد و برخلاف کاهش ناگهانی، شوک شدیدی به فرایند آموزش وارد نمی‌کند.

اگر learning rate را ثابت نگه داریم، در حالت بزرگ بودن آن، loss در مراحل پایانی بیشتر نوسان می‌کند و مدل ممکن است دائماً اطراف یک نقطه مناسب حرکت کند، بدون اینکه به‌خوبی همگرا شود. در این حالت نمودار loss معمولاً پرنوسان‌تر است و ممکن است در یک مقدار نسبتاً بالا متوقف شود.

اگر learning rate ثابت را خیلی کوچک انتخاب کنیم، نمودار loss نرم‌تر خواهد بود، اما کاهش آن بسیار آهسته می‌شود و آموزش اولیه زمان بیشتری می‌برد.

بنابراین cosine annealing کمک می‌کند در ابتدای آموزش پیشرفت سریع داشته باشیم و در انتهای آموزش با گام‌های کوچک‌تر به یک جواب پایدارتر برسیم.

</div>

## 8. Linear Evaluation

To assess the *quality* of the learned representations we follow the standard **linear-evaluation protocol**:

1. Load the pre-trained SimCLR checkpoint (`simclr_checkpoint.pth`).
2. Discard the projection head and bolt on a fresh **linear classifier** that maps the encoder output to `num_classes`.
3. **Freeze** all parameters of the backbone — only the new linear layer is trained.
4. Train on the labelled training split (STL-10 `train` or CIFAR-10) with `CrossEntropyLoss`.
5. Report top-1 and top-5 accuracy on the held-out test set.

High accuracy with a *frozen* encoder indicates that the contrastive objective produced semantically useful features.

In [12]:
# -------------------- Linear Evaluation --------------------
# Step 1 — labelled splits for the chosen dataset.
if dataset_name == 'stl10':
    train_data = datasets.STL10(
        './data',
        split='train',
        download=True,
        transform=transforms.ToTensor()
    )
    test_data = datasets.STL10(
        './data',
        split='test',
        download=True,
        transform=transforms.ToTensor()
    )
    num_classes = 10

elif dataset_name == 'cifar10':
    train_data = datasets.CIFAR10(
        './data',
        train=True,
        download=True,
        transform=transforms.ToTensor()
    )
    test_data = datasets.CIFAR10(
        './data',
        train=False,
        download=True,
        transform=transforms.ToTensor()
    )
    num_classes = 10

else:
    raise ValueError("Unsupported dataset for evaluation")


eval_train_loader = DataLoader(
    train_data,
    batch_size=256,
    shuffle=True,
    num_workers=2
)

eval_test_loader = DataLoader(
    test_data,
    batch_size=512,
    shuffle=False,
    num_workers=2
)


# Step 2 — load encoder, drop the projection head, attach a linear classifier.
def load_model_for_evaluation(
    checkpoint_path,
    base_model='resnet18',
    num_classes=10
):
    model = ResNetSimCLR(base_model=base_model, out_dim=128)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint['state_dict']

    # TODO 8.1 — Strip the projection-head weights from `state_dict`.
    # The projection head sits inside `backbone.fc.*` so delete every key
    # that startswith('backbone.fc').
    for k in list(state_dict.keys()):
        if k.startswith('backbone.fc'):
            del state_dict[k]

    log = model.load_state_dict(state_dict, strict=False)
    print("Missing keys when loading backbone:", log.missing_keys)

    # TODO 8.2 — Replace the (now random) projection head with a single
    # nn.Linear from dim_mlp -> num_classes.
    dim_mlp = model.backbone.fc[0].in_features
    model.backbone.fc = nn.Linear(dim_mlp, num_classes)

    return model.to(device)


eval_model = load_model_for_evaluation(
    'simclr_checkpoint.pth',
    'resnet18',
    num_classes
)

# TODO 8.3 — Freeze every parameter EXCEPT the new linear classifier.
# Hint: iterate over `eval_model.named_parameters()` and set
# `param.requires_grad = False` unless the name contains 'backbone.fc'.
for name, param in eval_model.named_parameters():
    param.requires_grad = 'backbone.fc' in name

trainable_params = [
    n for n, p in eval_model.named_parameters()
    if p.requires_grad
]

print("Trainable parameters:", trainable_params)

eval_optimizer = optim.Adam(
    eval_model.parameters(),
    lr=1e-3,
    weight_decay=1e-6
)

eval_criterion = nn.CrossEntropyLoss()


# TODO 8.4 — Train the linear head and evaluate after every epoch.
# For each epoch:
#   * loop over `eval_train_loader`, do a standard supervised optim step
#   * loop over `eval_test_loader` with no_grad and compute top-1 / top-5
#   * print "Epoch {e}: Loss=... | Test Top1=... | Test Top5=..."
eval_epochs = 20

for epoch in range(eval_epochs):
    # Keep the frozen backbone, including BatchNorm statistics, fixed.
    eval_model.eval()

    running_loss = 0.0
    train_samples = 0

    for images, targets in eval_train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        eval_optimizer.zero_grad(set_to_none=True)

        outputs = eval_model(images)
        loss = eval_criterion(outputs, targets)

        loss.backward()
        eval_optimizer.step()

        current_batch_size = targets.size(0)
        running_loss += loss.item() * current_batch_size
        train_samples += current_batch_size

    average_loss = running_loss / train_samples

    eval_model.eval()

    top1_correct = 0.0
    top5_correct = 0.0
    test_samples = 0

    with torch.no_grad():
        for images, targets in eval_test_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = eval_model(images)
            top1, top5 = accuracy(outputs, targets, topk=(1, 5))

            current_batch_size = targets.size(0)

            top1_correct += top1.item() * current_batch_size / 100.0
            top5_correct += top5.item() * current_batch_size / 100.0
            test_samples += current_batch_size

    test_top1 = 100.0 * top1_correct / test_samples
    test_top5 = 100.0 * top5_correct / test_samples

    print(
        f"Epoch {epoch + 1}: Loss={average_loss:.4f} | "
        f"Test Top1={test_top1:.2f}% | "
        f"Test Top5={test_top5:.2f}%"
    )

Missing keys when loading backbone: ['backbone.fc.0.weight', 'backbone.fc.0.bias', 'backbone.fc.2.weight', 'backbone.fc.2.bias']
Trainable parameters: ['backbone.fc.weight', 'backbone.fc.bias']
Epoch 1: Loss=2.0243 | Test Top1=46.11% | Test Top5=94.14%
Epoch 2: Loss=1.4856 | Test Top1=53.54% | Test Top5=95.46%
Epoch 3: Loss=1.2913 | Test Top1=57.11% | Test Top5=95.91%
Epoch 4: Loss=1.1947 | Test Top1=58.51% | Test Top5=96.40%
Epoch 5: Loss=1.1376 | Test Top1=59.06% | Test Top5=96.53%
Epoch 6: Loss=1.1027 | Test Top1=59.71% | Test Top5=96.72%
Epoch 7: Loss=1.0706 | Test Top1=60.41% | Test Top5=97.09%
Epoch 8: Loss=1.0495 | Test Top1=61.08% | Test Top5=96.83%
Epoch 9: Loss=1.0313 | Test Top1=60.29% | Test Top5=96.91%
Epoch 10: Loss=1.0166 | Test Top1=61.36% | Test Top5=97.31%
Epoch 11: Loss=1.0008 | Test Top1=61.83% | Test Top5=96.95%
Epoch 12: Loss=0.9997 | Test Top1=61.16% | Test Top5=96.91%
Epoch 13: Loss=0.9822 | Test Top1=61.35% | Test Top5=97.09%
Epoch 14: Loss=0.9722 | Test Top1=6

> **Question 11.** Why do we *freeze* the backbone during linear evaluation? What would happen — methodologically and empirically — if we instead fine-tuned the whole model end-to-end with the labels?

**Your answer to Q11:**

<div dir="rtl" align="right">


در ارزیابی خطی، backbone را ثابت نگه می‌داریم تا بررسی کنیم ویژگی‌هایی که مدل در مرحله contrastive pre-training یاد گرفته، به‌تنهایی تا چه اندازه مفید هستند. در این روش فقط یک classifier خطی آموزش داده می‌شود، بنابراین اگر دقت خوبی به دست بیاید، می‌توان نتیجه گرفت که کلاس‌های مختلف از قبل در فضای ویژگی‌های encoder به‌خوبی از یکدیگر جدا شده‌اند.

اگر اجازه دهیم تمام مدل با داده‌های برچسب‌دار fine-tune شود، وزن‌های encoder نیز تغییر می‌کنند. در این حالت دیگر مشخص نیست عملکرد خوب مدل به دلیل نمایش‌های یادگرفته‌شده در مرحله self-supervised بوده یا به دلیل آموزش دوباره encoder با استفاده از برچسب‌ها.

از نظر تجربی، fine-tuning کامل معمولاً دقت بیشتری نسبت به linear evaluation ایجاد می‌کند، چون همه لایه‌ها می‌توانند خودشان را با مسئله جدید تطبیق دهند. با این حال، به داده برچسب‌دار بیشتر، زمان آموزش بالاتر و تنظیم دقیق‌تر learning rate نیاز دارد و در مجموعه‌داده‌های کوچک ممکن است باعث overfitting شود.

بنابراین linear evaluation بیشتر برای اندازه‌گیری خالص کیفیت ویژگی‌های یادگرفته‌شده است، در حالی که fine-tuning برای رسیدن به بهترین عملکرد نهایی روی یک مسئله مشخص استفاده می‌شود.

</div>


## 9. Discussion

Reflect briefly (a few sentences each) on the following — these reflections count toward your grade.

- **Batch size.** How does the batch size affect the number of negatives and the training dynamics? At what point do you expect diminishing returns?
- **Projection head.** Re-state, in your own words, why the projection head helps during pre-training but is discarded for downstream use.
- **Augmentation ablation.** Try disabling **either** colour jitter **or** Gaussian blur and re-run a short pre-training + linear-eval. Report the change in linear-eval top-1 accuracy and discuss which augmentation matters more for your dataset.
- **Random baseline.** Compare the linear-eval accuracy to that of a randomly initialised (un-pretrained) backbone with the same linear head. How big is the gap?
- **Longer training.** If you have the compute, run pre-training for 100+ epochs and report the new linear-eval numbers.

### Further Reading

- Chen, T., Kornblith, S., Norouzi, M., & Hinton, G. (2020). *A Simple Framework for Contrastive Learning of Visual Representations*. ICML 2020.
- [SimCLR — official TF repo](https://github.com/google-research/simclr)
- [PyTorch SimCLR re-implementation](https://github.com/sthalles/SimCLR)
- [Exploring SimCLR — blog post](https://sthalles.github.io/simple-self-supervised-learning/)

<div dir="rtl" align="right">
اندازه Batch


در SimCLR، نمونه‌های دیگر موجود در batch به‌عنوان نمونه‌های منفی استفاده می‌شوند. بنابراین با بزرگ‌تر شدن batch size، تعداد نمونه‌های منفی بیشتر می‌شود و مدل مقایسه‌های متنوع‌تری در هر مرحله آموزش انجام می‌دهد. این موضوع معمولاً باعث بهتر شدن کیفیت ویژگی‌های یادگرفته‌شده می‌شود.

با این حال، افزایش بیش از حد batch size مصرف حافظه و زمان محاسباتی را زیاد می‌کند. همچنین بعد از رسیدن به تعداد کافی از نمونه‌های منفی، اضافه‌کردن نمونه‌های بیشتر احتمالاً بهبود خیلی کمی ایجاد می‌کند. در نتیجه، از یک نقطه به بعد بازده افزایش batch size کاهش پیدا می‌کند.

Projection Head

projection head در زمان pre-training کمک می‌کند contrastive loss در یک فضای جداگانه اعمال شود. به این ترتیب، encoder مجبور نیست تمام ویژگی‌های خود را مستقیماً با هدف contrastive loss سازگار کند و می‌تواند اطلاعات عمومی‌تر و مفیدتری از تصویر حفظ کند.

بعد از پایان pre-training، projection head کنار گذاشته می‌شود، چون خروجی encoder معمولاً برای مسائل downstream اطلاعات معنایی بیشتری دارد. بنابراین projection head بیشتر یک ابزار کمکی برای آموزش بهتر encoder است و حذف آن به معنی هدر رفتن آموزش نیست.

بررسی حذف Augmentation

در اجرای فعلی، آزمایش حذف colour jitter یا Gaussian blur انجام نشد، بنابراین نمی‌توانم تغییر دقیق top-1 accuracy را گزارش کنم.

با این حال، انتظار دارم حذف colour jitter روی CIFAR-10 تأثیر بیشتری داشته باشد، چون تصاویر این مجموعه کوچک هستند و مدل ممکن است به‌راحتی از رنگ و ظاهر سطحی برای شناسایی دو نمای یک تصویر استفاده کند. colour jitter مدل را مجبور می‌کند کمتر به رنگ وابسته باشد و بیشتر روی ساختار و شکل کلی تصویر تمرکز کند.

Gaussian blur نیز مفید است، اما به دلیل وضوح پایین تصاویر CIFAR-10، تاری شدید ممکن است بخشی از اطلاعات اصلی تصویر را حذف کند. برای مقایسه دقیق باید یک pre-training کوتاه بدون یکی از این دو augmentation اجرا شود و top-1 accuracy آن با نتیجه اصلی مقایسه شود.

مقایسه با Random Baseline

در اجرای فعلی، مدل pre-trained در مرحله linear evaluation به دقت top-1 حدود 62.34 درصد در epoch شانزدهم رسید. همچنین top-5 accuracy حدود 97.24 درصد بود.

آزمایش جداگانه با backbone کاملاً تصادفی انجام نشد، بنابراین مقدار دقیق random baseline در دسترس نیست. با این حال، انتظار می‌رود یک encoder تصادفی با classifier خطی عملکرد پایین‌تری داشته باشد، چون ویژگی‌های آن برای نمایش محتوای معنایی تصاویر آموزش ندیده‌اند.

اختلاف بین random baseline و نتیجه 62.34 درصدی نشان می‌دهد pre-training کنتراستی تا چه اندازه در یادگیری نمایش‌های مفید مؤثر بوده است. برای گزارش دقیق این اختلاف باید همان linear evaluation یک بار با ResNet تصادفی و بدون بارگذاری checkpoint اجرا شود.

آموزش طولانی‌تر

در این آزمایش، به دلیل محدودیت زمان و منابع محاسباتی، pre-training فقط برای یک اجرای کوتاه انجام شد و آموزش 100 یا بیشتر epoch اجرا نشد.

با وجود کوتاه بودن pre-training، linear evaluation به top-1 accuracy حدود 62.34 درصد رسید. انتظار می‌رود با افزایش تعداد epochهای pre-training، encoder ویژگی‌های پایدارتر و معنادارتری یاد بگیرد و دقت linear evaluation بیشتر شود.

البته میزان بهبود در طول زمان ثابت نیست و معمولاً بعد از تعداد زیادی epoch سرعت پیشرفت کمتر می‌شود. اجرای طولانی‌تر همچنین به تنظیم مناسب learning rate، scheduler و augmentationها نیاز دارد.

</div>